In [10]:
# BOGHAWATTA B P S
# IT22148254

import os
import json
import matplotlib.pyplot as plt
import numpy as np
import random
from typing import Tuple, List, Dict, Any

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm 
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    precision_recall_fscore_support, 
)
import torch.optim as optim

In [3]:
from utils.config import (
    MODEL_NAME, NUM_CLASSES, BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS, DEVICE,
    LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS, T_MAX_LR_SCHEDULER_EPOCHS, CHECKPOINT_PATH,
    TRAIN_DIR, TEST_DIR, VAL_DIR, CHECKPOINT_PATH_1, FREEZE_EPOCHS, FINE_TUNE_LR, PRECISION_BOOST_FACTOR
)

In [4]:
from utils.dataset import ChestXrayDataset, train_tf, val_tf, get_file_paths_and_labels

#### Set seeds for reproducibility

In [5]:
def set_seed(seed: int = 42) -> None:
    """Sets the seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # For deterministic behavior
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False
    print(f"Seeds set to {seed}.")

#### -------------------- Model Definition --------------------

In [6]:
def get_model(model_name: str, num_classes: int, pretrained: bool = True, freeze_base: bool = False) -> nn.Module:
    """Instantiates a Vision Transformer (ViT) model from timm and optionally freezes its base."""
    
    model = timm.create_model(
        model_name, 
        pretrained=pretrained, 
        num_classes=num_classes
    )
    
    if freeze_base:
        # Freeze all parameters except those in the classification head ('head' in timm's ViT)
        for name, param in model.named_parameters():
             if 'head' not in name:
                 param.requires_grad = False
             
    print(f"Model: {model_name} instantiated. Number of classes: {num_classes}. Base Frozen: {freeze_base}")
    return model

### -------------------- Metrics logging for visualisations --------------------------------
#####  Score categories : train_loss, train_acc, val_loss, val_acc, val_auc, val_prec, val_rec, val_f1

In [7]:
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_auc': [],
    'val_prec': [],
    'val_rec': [],
    'val_f1': []
}

##### --- NEW: Function to Save History ---

In [8]:
def save_history(history: dict, filename: str = CHECKPOINT_PATH_1 + 'training_history.json', output_dir: str = '.'):
    """Saves the training history dictionary to a JSON file."""
    try:
        # Convert NumPy float64 types to standard Python float for JSON serialization
        history_serializable = {
            k: [float(v_item) for v_item in v] 
            for k, v in history.items()
        }
        
        filepath = os.path.join(output_dir, filename)
        with open(filepath, 'w') as f:
            json.dump(history_serializable, f, indent=4)
        print(f"✅ Training history saved to {filepath}")
    except Exception as e:
        print(f"Error saving history: {e}")

##### --- Function to Load History ---

In [11]:
def load_history(filename: str = 'vit_model_history.json', input_dir: str = '.') -> Dict[str, Any]:
    """Loads the training history dictionary from a JSON file."""
    filepath = os.path.join(input_dir, filename)
    try:
        with open(filepath, 'r') as f:
            history = json.load(f)
        print(f"✅ Training history loaded from {filepath}")
        return history
    except FileNotFoundError:
        print(f"Error: History file not found at {filepath}")
        return {}
    except Exception as e:
        print(f"Error loading history: {e}")
        return {}

##### --- Plotting Function ---

In [12]:
from ViT.utils.config import PLOTS_OUTPUT_DIR


def plot_training_history(history: dict, epochs: int, output_dir: str = PLOTS_OUTPUT_DIR):
    """Generates and saves plots for Loss, Accuracy, AUC, and F1-Score history to a folder."""
    
    # 1. Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created output directory: {output_dir}")
        
    epochs_range = range(1, epochs + 1)
    
    # Define a consistent plotting function
    def plot_metric(metric_name, y_train, y_val, title):
        plt.figure(figsize=(10, 6))
        plt.plot(epochs_range, y_train, label=f'Training {metric_name}', marker='.', linestyle='-')
        plt.plot(epochs_range, y_val, label=f'Validation {metric_name}', marker='.', linestyle='-')
        plt.title(title)
        plt.xlabel('Epoch')
        plt.ylabel(metric_name)
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        
        # Save plot to the specified output_dir
        filename = f'{metric_name.lower().replace("-", "_")}_{CHECKPOINT_PATH_1}_history.png'
        plt.savefig(os.path.join(output_dir, filename))
        plt.close()

    # 1. Loss History
    plot_metric('Loss', history['train_loss'], history['val_loss'], 'Loss Over Epochs (Indicator of Overfitting)')

    # 2. Accuracy History
    plot_metric('Accuracy', history['train_acc'], history['val_acc'], 'Accuracy Over Epochs')
    
    # 3. AUC History (Validation only)
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, history['val_auc'], label='Validation AUC', marker='.', linestyle='-')
    plt.title('AUC Over Epochs (Model Discriminative Power)')
    plt.xlabel('Epoch')
    plt.ylabel('AUC')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    filename = 'auc_history.png'
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

    # 4. F1-Score History 
    # Since F1 is a Validation metric, we use validation data for both lines in the plot. 
    plot_metric('F1-Score', history['val_f1'], history['val_f1'], 'F1-Score Over Epochs (Balanced Performance)')
    
    print(f"\nTraining history plots generated and saved to the '{output_dir}' folder.")

#### -------------------- Training and Evaluation Functions --------------------
##### 1. Training

In [13]:
def train_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    optimizer: torch.optim.Optimizer, 
    criterion: nn.Module, 
    device: str,
    scaler: torch.cuda.amp.GradScaler,
    scheduler: optim.lr_scheduler._LRScheduler = None 
) -> Tuple[float, float]:
    """Runs a single training epoch."""
    model.train()
    losses: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    # Use enumerate for tracking progress more clearly if needed, but tqdm is sufficient
    for images, labels in tqdm(loader, desc="Training"):
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed Precision Training
        with torch.autocast(device_type=device, dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if scheduler is not None:
             scheduler.step()
        
        # Metrics collection
        losses.append(loss.item())
        
        # Use .detach().cpu() only when necessary for non-gradient operations
        preds = torch.argmax(logits.detach().cpu(), dim=1).numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy()) # labels are already on device, move back for numpy
        
    acc = accuracy_score(all_labels, all_preds)
    return float(np.mean(losses)), float(acc)

##### 2. Evaluating

In [14]:
def eval_epoch(
    model: nn.Module, 
    loader: DataLoader, 
    criterion: nn.Module, 
    device: str
) -> Dict[str, float]:
    """Runs a single evaluation epoch and returns comprehensive metrics."""
    model.eval()
    losses: List[float] = []
    all_probs: List[float] = []
    all_preds: List[int] = []
    all_labels: List[int] = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)
            
            with torch.autocast(device_type=device, dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
                # Softmax to get probabilities for AUC, choosing the positive class (index 1)
                probs = torch.softmax(logits, dim=1)[:, 1] 
                
            losses.append(loss.item())
            
            # Metrics collection
            preds = torch.argmax(logits.cpu(), dim=1).numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
    # Calculate comprehensive metrics
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    
    try:
        # AUC requires probabilities for the positive class
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        # Happens if only one class is present in the batch/dataset (rare, but good to handle)
        auc = 0.0
        
    # precision, recall, f1 for binary classification
    prec, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='binary', zero_division=0
    )
    
    # Optional: Confusion Matrix
    # cm = confusion_matrix(all_labels, all_preds)
    
    return {
        'loss': avg_loss,
        'accuracy': acc,
        'auc': auc,
        'precision': prec,
        'recall': recall,
        'f1': f1,
    }

### -------------------- Main Training Loop --------------------

##### 1. Data Preparation

In [15]:
set_seed(42)
try:
    train_files, train_labels, train_weights_np = get_file_paths_and_labels(TRAIN_DIR)
    # val_files, val_labels, _ = get_file_paths_and_labels(VAL_DIR)
    
    boosted_weights_np = train_weights_np.copy()
    boosted_weights_np[0] *= PRECISION_BOOST_FACTOR
    
    val_files, val_labels, _ = get_file_paths_and_labels(TEST_DIR)
    
except FileNotFoundError as e:
    print(f"Error: Data directory not found. Please update BASE_DIR in config.py.")
    print(f"Missing directory: {e}")

Seeds set to 42.
Loaded 5216 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\train. Class counts: {'NORMAL': 1341, 'PNEUMONIA': 3875}
Calculated class weights: [1.9448173  0.67303226] (Index 0: NORMAL, Index 1: PNEUMONIA)
Loaded 624 samples from C:\Users\HP\Desktop\SLIIT\Y4 SEM 1\DL\Ass\Assignment\DL-project\ViT\dataset\chest_xray\test. Class counts: {'NORMAL': 234, 'PNEUMONIA': 390}
Calculated class weights: [1.33333333 0.8       ] (Index 0: NORMAL, Index 1: PNEUMONIA)


##### 2. Datasets and DataLoaders

In [16]:
train_ds = ChestXrayDataset(train_files, train_labels, transform=train_tf)
val_ds   = ChestXrayDataset(val_files, val_labels, transform=val_tf)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
val_loader   = DataLoader(
    val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)
print("DataLoaders initialized.")

DataLoaders initialized.


### 3. Training phases
##### Phase 1 

In [17]:
# --- Phase 1: Train ONLY the Head (for 5 epochs) ---
FULL_EPOCHS = NUM_EPOCHS - FREEZE_EPOCHS

# 1. Instantiate the model with the base FROZEN
print(f"\n--- Starting Phase 1: Frozen Base (Head Only) for {FREEZE_EPOCHS} Epochs ---")
model = get_model(MODEL_NAME, NUM_CLASSES, freeze_base=True).to(DEVICE)


--- Starting Phase 1: Frozen Base (Head Only) for 5 Epochs ---
Model: vit_base_patch16_224 instantiated. Number of classes: 2. Base Frozen: True


##### Configurations for phase 1 : Optimiser, Scaler, Weights, Scheduler

In [18]:
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), 
    lr=LEARNING_RATE, 
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(device=DEVICE)  # Initialize scaler once

best_val_auc = 0.0 # Best AUC tracker

# class_weights = torch.tensor(train_weights_np, dtype=torch.float32).to(DEVICE) 
class_weights = torch.tensor(boosted_weights_np, dtype=torch.float32).to(DEVICE) 
criterion = nn.CrossEntropyLoss(weight=class_weights)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=T_MAX_LR_SCHEDULER_EPOCHS * len(train_loader) # T_max is number of steps
) 

In [19]:
print(f"\n--- Starting Phase 1: Frozen Base (Head Only) for {FREEZE_EPOCHS} Epochs ---")

for epoch in range(1, FREEZE_EPOCHS + 1):
    # TRAIN
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
        
    # VALIDATE
    val_metrics = eval_epoch(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = (
        val_metrics['loss'], val_metrics['accuracy'], val_metrics['auc'], 
        val_metrics['precision'], val_metrics['recall'], val_metrics['f1']
    )
        
    # SCHEDULER is not needed here
    
    # Metrics adding to history (for plots)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_prec'].append(val_metrics['precision'])
    history['val_rec'].append(val_metrics['recall'])
    history['val_f1'].append(val_metrics['f1'])
        
    # LOGGING
    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, "
        f"Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}"
    )
        
    # SAVE BEST MODEL
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_auc': best_val_auc}, CHECKPOINT_PATH_1)
        print(f"--- Model saved! New best AUC: {best_val_auc:.4f} ---")
        # Or CHECKPOINT_PATH as prev 



--- Starting Phase 1: Frozen Base (Head Only) for 5 Epochs ---


Validating: 100%|██████████| 20/20 [00:09<00:00,  2.10it/s]


Epoch 1/30 | Train Loss: 0.4584, Train Acc: 0.8227 | Val Loss: 0.5678, Val Acc: 0.7997, Val AUC: 0.9242, Val Recall: 0.9795, Val F1: 0.8594
--- Model saved! New best AUC: 0.9242 ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.18it/s]


Epoch 2/30 | Train Loss: 0.2936, Train Acc: 0.9061 | Val Loss: 0.5308, Val Acc: 0.8446, Val AUC: 0.9277, Val Recall: 0.9718, Val F1: 0.8865
--- Model saved! New best AUC: 0.9277 ---


...

Epoch 3/30 | Train Loss: 0.2437, Train Acc: 0.9149 | Val Loss: 0.5439, Val Acc: 0.8349, Val AUC: 0.9304, Val Recall: 0.9692, Val F1: 0.8801
--- Model saved! New best AUC: 0.9304 ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.21it/s]


Epoch 4/30 | Train Loss: 0.2154, Train Acc: 0.9227 | Val Loss: 0.6622, Val Acc: 0.7949, Val AUC: 0.9303, Val Recall: 0.9846, Val F1: 0.8571


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.14it/s]

Epoch 5/30 | Train Loss: 0.2010, Train Acc: 0.9266 | Val Loss: 0.6507, Val Acc: 0.8173, Val AUC: 0.9269, Val Recall: 0.9795, Val F1: 0.8702


### Phase 2
##### Unfreezing and fine-tuning

In [20]:
# --- Phase 2: Unfreeze and Fine-Tune the Whole Model ---
print(f"\n--- Transitioning to Phase 2: Unfreezing All Layers ---")

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

# Re-define Optimizer with the new (lower) Fine-Tune LR for all parameters
optimizer = optim.AdamW(model.parameters(), lr=FINE_TUNE_LR, weight_decay=WEIGHT_DECAY)

# Use CosineAnnealingLR for smooth decay during fine-tuning
TOTAL_STEPS = FULL_EPOCHS * len(train_loader) 
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_STEPS
) 


--- Transitioning to Phase 2: Unfreezing All Layers ---


In [21]:
print(f"--- Starting Phase 2: Fine-Tuning All Layers for {FULL_EPOCHS} Epochs ---")

for epoch in range(FREEZE_EPOCHS + 1, NUM_EPOCHS + 1):
     # TRAIN
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler, scheduler)
        
    # VALIDATE
    val_metrics = eval_epoch(model, val_loader, criterion, DEVICE)
    val_loss, val_acc, val_auc, val_prec, val_rec, val_f1 = (
        val_metrics['loss'], val_metrics['accuracy'], val_metrics['auc'], 
        val_metrics['precision'], val_metrics['recall'], val_metrics['f1']
    )
        
    # SCHEDULER is called inside train epoch since we used TOTAL_STEPS for T_max (per batch)
     
    # Metrics adding to history (for plots)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)
    history['val_prec'].append(val_metrics['precision'])
    history['val_rec'].append(val_metrics['recall'])
    history['val_f1'].append(val_metrics['f1'])
        
    # LOGGING
    print(
        f"Epoch {epoch}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}, "
        f"Val Recall: {val_rec:.4f}, Val F1: {val_f1:.4f}"
    )
        
    # SAVE BEST MODEL
    if val_metrics['auc'] > best_val_auc:
        best_val_auc = val_metrics['auc']
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'best_val_auc': best_val_auc}, CHECKPOINT_PATH_1)
        print(f"--- Model saved! New best AUC: {best_val_auc:.4f} ---")  
        # Or CHECKPOINT_PATH as prev 
         
# save history after all the epochs are over
save_history(history, filename=CHECKPOINT_PATH_1 + 'vit_model_history.json') 

--- Starting Phase 2: Fine-Tuning All Layers for 25 Epochs ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.15it/s]


Epoch 6/30 | Train Loss: 0.1728, Train Acc: 0.9329 | Val Loss: 0.5013, Val Acc: 0.8942, Val AUC: 0.9705, Val Recall: 0.9949, Val F1: 0.9216
--- Model saved! New best AUC: 0.9705 ---


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.14it/s]


Epoch 7/30 | Train Loss: 0.0963, Train Acc: 0.9630 | Val Loss: 1.7312, Val Acc: 0.7660, Val AUC: 0.9663, Val Recall: 1.0000, Val F1: 0.8423


...

Epoch 28/30 | Train Loss: 0.0039, Train Acc: 0.9985 | Val Loss: 2.6619, Val Acc: 0.8013, Val AUC: 0.9716, Val Recall: 1.0000, Val F1: 0.8628


Validating: 100%|██████████| 20/20 [00:03<00:00,  5.23it/s]


Epoch 29/30 | Train Loss: 0.0038, Train Acc: 0.9983 | Val Loss: 2.8219, Val Acc: 0.7965, Val AUC: 0.9701, Val Recall: 1.0000, Val F1: 0.8600


Validating: 100%|██████████| 20/20 [00:04<00:00,  4.91it/s]

Epoch 30/30 | Train Loss: 0.0027, Train Acc: 0.9988 | Val Loss: 2.8171, Val Acc: 0.7965, Val AUC: 0.9703, Val Recall: 1.0000, Val F1: 0.8600
✅ Training history saved to .\vit_chest_xray_best_1.pthvit_model_history.json


In [22]:
loaded_history = load_history(filename = CHECKPOINT_PATH_1 + 'vit_model_history.json')

plot_training_history(history, NUM_EPOCHS)

✅ Training history loaded from .\vit_chest_xray_best_1.pthvit_model_history.json
Created output directory: plots_vit

Training history plots generated and saved to the 'plots_vit' folder.
